# M1: Instrument & Optimize a LangGraph Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/OpenTrace/blob/main/examples/notebooks/01_m1_instrument_and_optimize.ipynb)

This notebook demonstrates the **M1 core value proposition**: drop-in OTEL
instrumentation and end-to-end optimization for any LangGraph agent.

## What this notebook proves

| Gate | Verified |
|------|----------|
| `instrument_graph()` wraps a LangGraph with OTEL tracing | Section 4 |
| `param.*` + `param.*.trainable` attributes on spans | Section 5 |
| OTLP → TGJ → `ParameterNode` + `MessageNode` | Section 6 |
| Child spans do NOT break temporal chaining | Section 6 |
| `apply_updates()` changes prompt templates via bindings | Section 7 |
| `optimize_graph()` full loop (StubLLM — deterministic) | Section 8 |
| `optimize_graph()` live provider (OpenRouter, guarded) | Section 9 |

## Modes

- **StubLLM mode** (Sections 4-8): runs without any API keys — deterministic, CI-safe.
- **Live LLM mode** (Section 9): requires `OPENROUTER_API_KEY` via Colab Secrets or `.env`.

## Table of Contents

1. [Install Dependencies](#1-install-dependencies)
2. [Configuration](#2-configuration)
3. [Define a Minimal LangGraph](#3-define-a-minimal-langgraph)
4. [Instrument the Graph (StubLLM)](#4-instrument-the-graph-stubllm)
5. [Inspect OTLP Spans & param.* Attributes](#5-inspect-otlp-spans--param-attributes)
6. [OTLP → TGJ → Trace Nodes](#6-otlp--tgj--trace-nodes)
7. [Bindings & apply_updates()](#7-bindings--apply_updates)
8. [optimize_graph() — StubLLM End-to-End](#8-optimize_graph--stubllm-end-to-end)
9. [Live LLM Mode (OpenRouter)](#9-live-llm-mode-openrouter)
10. [Save Artifacts](#10-save-artifacts)

---
## 1. Install Dependencies

Run this cell once to install all required packages.

In [1]:
!pip install -q langgraph>=1.0.0 opentelemetry-api>=1.38.0 opentelemetry-sdk>=1.38.0 \
    python-dotenv>=1.0.0 requests>=2.28.0 typing_extensions>=4.0.0 graphviz>=0.20.1

# Install OpenTrace (the project itself) in editable mode
# If running on Colab, install from the repo and checkout OPENTRACE_REF
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True

    OPENTRACE_REPO = "https://github.com/AgentOpt/OpenTrace.git"
    OPENTRACE_REF = os.environ.get("OPENTRACE_REF", "main")

    if not os.path.exists("/content/OpenTrace"):
        !git clone {OPENTRACE_REPO} /content/OpenTrace
    !git -C /content/OpenTrace checkout {OPENTRACE_REF}
    !pip install -q -e /content/OpenTrace

    print(f"[INFO] OpenTrace ref: {OPENTRACE_REF}")
except ImportError:
    IN_COLAB = False
    # Assume local dev: project already installed via pip install -e .

print("\n" + "=" * 50)
print("All dependencies installed!")
print("=" * 50)


All dependencies installed!


**Persistent output (Colab):** When running on Colab the next cell mounts
Google Drive so artifacts survive session restarts. Locally they go into
`./notebook_outputs/`.

In [2]:
import os
from datetime import datetime

RUN_FOLDER = None
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OPENTRACE_REF = os.environ.get("OPENTRACE_REF", "main")
    base = f"/content/drive/MyDrive/OpenTrace_runs/M1/{OPENTRACE_REF}"
    os.makedirs(base, exist_ok=True)
    RUN_FOLDER = os.path.join(base, f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (Google Drive, OpenTrace): {RUN_FOLDER}")
except Exception:
    RUN_FOLDER = os.path.abspath(os.path.join(os.getcwd(), "notebook_outputs", "m1"))
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (local): {RUN_FOLDER}")

---
## 2. Configuration

API keys are retrieved **automatically** — never paste keys into cells:

| Priority | Source | How to set |
|----------|--------|------------|
| 1 | **Colab Secrets** | Click the key icon → add `OPENROUTER_API_KEY` |
| 2 | **Environment variable** | `export OPENROUTER_API_KEY=sk-or-v1-...` |
| 3 | **`.env` file** | `OPENROUTER_API_KEY=sk-or-v1-...` in project root |

Sections 4-8 use **StubLLM** (no key needed). Section 9 uses a live
provider and is skipped automatically when no key is available.

In [3]:
from __future__ import annotations
import os, json

# Model config (free tier on OpenRouter)
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct:free"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Budget guard for live mode
MAX_TOKENS_PER_CALL = 256
LIVE_TEMPERATURE = 0  # deterministic

# ---------- key retrieval (Colab Secrets → env → .env file) ----------
OPENROUTER_API_KEY = ""

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    pass

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from environment variable.")

if not OPENROUTER_API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
        if OPENROUTER_API_KEY:
            print("[INFO] API key loaded from .env file.")
    except ImportError:
        pass

HAS_API_KEY = bool(OPENROUTER_API_KEY)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print(f"\nAPI key: {'[SET]' if HAS_API_KEY else '[NOT SET — live mode will be skipped]'}")
print(f"Model:   {OPENROUTER_MODEL}")
print(f"Budget:  max_tokens={MAX_TOKENS_PER_CALL}, temperature={LIVE_TEMPERATURE}")

[INFO] API key loaded from .env file.

API key: [SET]
Model:   meta-llama/llama-3.1-8b-instruct:free
Budget:  max_tokens=256, temperature=0


---
## 3. Define a Minimal LangGraph

A simple **planner → synthesizer** pipeline. Node functions close over
`tracing_llm` and `templates` so that `apply_updates()` propagates to
the next invocation automatically.

In [ ]:
from typing import Any, Dict, List, Optional
from typing_extensions import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# Keep the notebook aligned with JSON_OTEL_trace_optim_demo_LANGGRAPH.py
DEMO_QUERIES = [
    "Summarize the causes and key events of the French Revolution.",
    "Give 3 factual relationships about Tesla, Inc. with entity IDs.",
    "What is the Wikidata ID for CRISPR and list 2 related entities?",
]

class AgentState(TypedDict, total=False):
    query: str
    plan: Dict[str, Any]
    current_step: int
    contexts: List[str]
    agent_query: str
    final_answer: str
    eval_score: float
    eval_feedback: str

def wikipedia_search(query: str) -> str:
    """Wikipedia tool. Falls back gracefully if wikipedia package/network is unavailable."""
    try:
        import wikipedia
        wikipedia.set_lang("en")
        hits = wikipedia.search(query, results=2)
        out = []
        for h in hits:
            try:
                s = wikipedia.summary(h, sentences=3, auto_suggest=False, redirect=True)
                out.append(f"### {h}\n{s}")
            except Exception:
                continue
        return "\n\n".join(out) or "No Wikipedia results."
    except Exception:
        return "Wikipedia search unavailable."

def wikidata_search(query: str) -> str:
    """Wikidata search tool (wbsearchentities)."""
    import requests
    try:
        r = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={
                "action": "wbsearchentities",
                "format": "json",
                "language": "en",
                "search": query[:100],
                "limit": 5,
            },
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()
        results = [
            f"- {item.get('label','')}: {item.get('description','')} ({item.get('id','')})"
            for item in data.get("search", [])
        ]
        return "\n".join(results) if results else "No Wikidata entities found."
    except Exception:
        return f"Wikidata search unavailable. Query: {query[:50]}..."

def build_graph(tracing_llm, templates: Dict[str, str]):
    """
    Build a multi-node LangGraph aligned with JSON_OTEL_trace_optim_demo_LANGGRAPH.py:
    planner -> executor -> (web_researcher|wikidata_researcher|synthesizer) -> evaluator
    """

    def planner_node(state: AgentState) -> Command[Literal["executor"]]:
        template = templates.get(
            "planner_prompt",
            "Return JSON plan with steps for query: {query}. Use agents: web_researcher, wikidata_researcher, synthesizer.",
        )
        prompt = template.replace("{query}", state.get("query", ""))

        raw = tracing_llm.node_call(
            span_name="planner",
            template_name="planner_prompt",
            template=template,
            optimizable_key="planner",
            user_query=state.get("query", ""),
            extra_inputs={"user_query": state.get("query", "")},
            messages=[
                {"role": "system", "content": "Return JSON only. Keys: 1,2,... each step has {agent,action,goal,query}."},
                {"role": "user", "content": prompt},
            ],
            max_tokens=400,
            temperature=0,
        )
        plan: Dict[str, Any]
        try:
            import json
            plan = json.loads(raw)
        except Exception:
            q = (state.get("query", "") or "").lower()
            plan = {
                "1": {"agent": "web_researcher", "action": "search", "goal": "collect context", "query": state.get("query", "")},
                "2": {"agent": "wikidata_researcher" if ("wikidata" in q or "entity id" in q or "id" in q) else "synthesizer",
                      "action": "search" if ("wikidata" in q or "entity id" in q or "id" in q) else "answer",
                      "goal": "entities or final answer", "query": state.get("query", "")},
                "3": {"agent": "synthesizer", "action": "answer", "goal": "final answer", "query": state.get("query", "")},
            }

        return Command(update={"plan": plan, "current_step": 1, "contexts": []}, goto="executor")

    def executor_node(state: AgentState) -> Command[Literal["web_researcher", "wikidata_researcher", "synthesizer"]]:
        step = int(state.get("current_step", 1) or 1)
        plan = state.get("plan", {}) or {}
        plan_step = plan.get(str(step), {})
        if not plan_step:
            return Command(update={}, goto="synthesizer")

        template = templates.get(
            "executor_prompt",
            "Given step {step} of plan: {plan_step}\nFor query: {query}\nReturn JSON: {goto,query}. goto in [web_researcher,wikidata_researcher,synthesizer].",
        )
        prompt = (
            template.replace("{step}", str(step))
            .replace("{plan_step}", str(plan_step))
            .replace("{query}", state.get("query", ""))
        )

        raw = tracing_llm.node_call(
            span_name="executor",
            template_name="executor_prompt",
            template=template,
            optimizable_key="executor",
            user_query=state.get("query", ""),
            extra_inputs={"step": str(step), "user_query": state.get("query", "")},
            messages=[
                {"role": "system", "content": "Return JSON only with keys goto and query."},
                {"role": "user", "content": prompt},
            ],
            max_tokens=200,
            temperature=0,
        )

        goto = str(plan_step.get("agent", "synthesizer"))
        q2 = str(plan_step.get("query", state.get("query", "")))
        try:
            import json
            d = json.loads(raw)
            goto = str(d.get("goto", goto))
            q2 = str(d.get("query", q2))
        except Exception:
            pass

        if goto not in ("web_researcher", "wikidata_researcher", "synthesizer"):
            goto = "synthesizer"

        return Command(update={"agent_query": q2}, goto=goto)

    def web_researcher_node(state: AgentState) -> Command[Literal["executor"]]:
        q = state.get("agent_query", state.get("query", ""))
        with tracing_llm.tracer.start_as_current_span("web_researcher") as sp:
            sp.set_attribute("inputs.user_query", state.get("query", ""))
            sp.set_attribute("inputs.agent_query", q)
            ctx = wikipedia_search(q)
            sp.set_attribute("outputs.context.preview", (ctx or "")[:500])
        contexts = list(state.get("contexts", []) or [])
        contexts.append(ctx)
        step = int(state.get("current_step", 1) or 1) + 1
        return Command(update={"contexts": contexts, "current_step": step}, goto="executor")

    def wikidata_researcher_node(state: AgentState) -> Command[Literal["executor"]]:
        q = state.get("agent_query", state.get("query", ""))
        with tracing_llm.tracer.start_as_current_span("wikidata_researcher") as sp:
            sp.set_attribute("inputs.user_query", state.get("query", ""))
            sp.set_attribute("inputs.agent_query", q)
            ctx = wikidata_search(q)
            sp.set_attribute("outputs.context.preview", (ctx or "")[:500])
        contexts = list(state.get("contexts", []) or [])
        contexts.append(ctx)
        step = int(state.get("current_step", 1) or 1) + 1
        return Command(update={"contexts": contexts, "current_step": step}, goto="executor")

    def synthesizer_node(state: AgentState) -> Command[Literal["evaluator"]]:
        template = templates.get(
            "synthesizer_prompt",
            "Answer the query: {query}\nContext:\n{contexts}\nIf asked for IDs, include them. Be factual.",
        )
        contexts = "\n\n".join(state.get("contexts", []) or [])
        prompt = template.replace("{query}", state.get("query", "")).replace("{contexts}", contexts[:4000])

        ans = tracing_llm.node_call(
            span_name="synthesizer",
            template_name="synthesizer_prompt",
            template=template,
            optimizable_key="synthesizer",
            user_query=state.get("query", ""),
            extra_inputs={"user_query": state.get("query", "")},
            messages=[
                {"role": "system", "content": "You are a careful assistant."},
                {"role": "user", "content": prompt},
            ],
            max_tokens=500,
            temperature=0,
        )
        return Command(update={"final_answer": ans}, goto="evaluator")

    def evaluator_node(state: AgentState) -> Command[Literal["__end__"]]:
        import re
        q = (state.get("query", "") or "").lower()
        ans = (state.get("final_answer", "") or "")
        ctx = "\n".join(state.get("contexts", []) or "")
        wants_ids = ("wikidata" in q) or ("entity id" in q) or ("id" in q and "tesla" in q)
        has_qid = bool(re.search(r"\bQ\d{2,}\b", ans))
        uses_ctx = len(ctx.strip()) > 0
        score = 0.25
        reasons = []
        if uses_ctx:
            score += 0.25
            reasons.append("uses_context")
        if wants_ids and has_qid:
            score += 0.35
            reasons.append("has_qids")
        if "french revolution" in q and len(ans) > 200:
            score += 0.10
            reasons.append("sufficient_detail")
        score = min(score, 0.95)
        feedback = {"score": score, "reasons": reasons}
        with tracing_llm.tracer.start_as_current_span("evaluator") as sp:
            sp.set_attribute("eval.score", str(score))
            sp.set_attribute("eval.reasons", ",".join(reasons))
        return Command(update={"eval_score": score, "eval_feedback": str(feedback)}, goto=END)

    workflow = StateGraph(AgentState)
    workflow.add_node("planner", planner_node)
    workflow.add_node("executor", executor_node)
    workflow.add_node("web_researcher", web_researcher_node)
    workflow.add_node("wikidata_researcher", wikidata_researcher_node)
    workflow.add_node("synthesizer", synthesizer_node)
    workflow.add_node("evaluator", evaluator_node)

    workflow.add_edge(START, "planner")
    workflow.add_edge("synthesizer", "evaluator")

    return workflow.compile()

print("Graph builder defined.")
print(f"  Nodes: planner, executor, web_researcher, wikidata_researcher, synthesizer, evaluator")
print(f"  DEMO_QUERIES: {len(DEMO_QUERIES)} queries")

### StubLLM

A deterministic LLM that returns canned responses (no API calls).

In [ ]:
class StubLLM:
    """Deterministic LLM stub for the multi-node graph.

    Produces JSON plans for planner, routing JSON for executor,
    and text answers for synthesizer. When the prompt template includes
    optimization signals ("step-by-step", "thorough"), the stub produces
    richer plans and more detailed answers so eval_fn returns a higher
    score — demonstrating non-saturating optimization.
    """
    model = "stub-llm"

    def __init__(self):
        self.call_count = 0

    def __call__(self, messages=None, **kwargs):
        self.call_count += 1
        import json as _json

        content = f"Stub response #{self.call_count}"
        if messages:
            user_text = ""
            system_text = ""
            for m in messages:
                if m.get("role") == "user":
                    user_text = (m.get("content") or "").lower()
                elif m.get("role") == "system":
                    system_text = (m.get("content") or "").lower()

            # Detect if the prompt template has been optimized
            is_enhanced = any(kw in user_text for kw in ("step-by-step", "thorough", "detailed"))

            if "return json only" in system_text and "step" in system_text:
                # Planner: return a JSON plan
                q = user_text
                if is_enhanced:
                    # Optimized prompt -> richer plan with more steps
                    steps = {
                        "1": {"agent": "web_researcher", "action": "search", "goal": "gather primary context", "query": user_text[:80]},
                        "2": {"agent": "wikidata_researcher", "action": "search", "goal": "find entity IDs", "query": user_text[:80]},
                        "3": {"agent": "web_researcher", "action": "verify", "goal": "cross-check facts", "query": user_text[:80]},
                        "4": {"agent": "synthesizer", "action": "answer", "goal": "produce comprehensive answer", "query": user_text[:80]},
                    }
                else:
                    # Baseline prompt -> simpler plan
                    steps = {
                        "1": {"agent": "web_researcher", "action": "search", "goal": "collect context", "query": user_text[:80]},
                        "2": {"agent": "synthesizer", "action": "answer", "goal": "final answer", "query": user_text[:80]},
                    }
                content = _json.dumps(steps)

            elif "return json only" in system_text and "goto" in system_text:
                # Executor: return routing JSON
                content = _json.dumps({"goto": "synthesizer", "query": user_text[:80]})

            elif "careful assistant" in system_text:
                # Synthesizer: return a text answer
                if "french revolution" in user_text:
                    content = (
                        "The French Revolution (1789-1799) was caused by fiscal crisis, social inequality, "
                        "and Enlightenment ideas. Key events include the Storming of the Bastille (July 14, 1789), "
                        "the Declaration of the Rights of Man, the Reign of Terror, and Napoleon's rise to power."
                    )
                elif "tesla" in user_text:
                    content = (
                        "Tesla, Inc. (Q478214) is an American electric vehicle manufacturer. "
                        "Key relationships: 1) Founded by Elon Musk (Q317521). "
                        "2) Headquartered in Austin, Texas (Q16559). "
                        "3) Produces the Model S, Model 3, Model X, and Model Y vehicles."
                    )
                elif "crispr" in user_text:
                    content = (
                        "CRISPR (Q22328579) is a gene-editing technology. "
                        "Related entities: 1) Cas9 protein (Q24721710) - the molecular scissors. "
                        "2) Jennifer Doudna (Q467958) - co-discoverer of CRISPR-Cas9."
                    )
                else:
                    content = f"Based on the collected context, here is a comprehensive answer about the topic."
            else:
                content = f"Stub response #{self.call_count}: Generic LLM output."

        class _Msg:
            pass
        msg = _Msg()
        msg.content = content
        class _Choice:
            pass
        choice = _Choice()
        choice.message = msg
        class _Resp:
            pass
        resp = _Resp()
        resp.choices = [choice]
        return resp

stub_llm = StubLLM()
print("StubLLM ready (multi-node graph aware, prompt-template-sensitive).")

---
## 4. Instrument the Graph (StubLLM)

One function call — `instrument_graph()` — wraps the LangGraph with full
OTEL tracing, creates a `TelemetrySession`, and sets up `Binding` objects
that map `param.*` keys to the live template dict.

In [ ]:
from opto.trace.io import instrument_graph, apply_updates

INITIAL_TEMPLATES = {
    "planner_prompt":      "Create a JSON plan for: {query}. Use web_researcher and synthesizer; include wikidata_researcher if IDs are requested.",
    "executor_prompt":     "Given step {step} of plan: {plan_step} for query: {query}. Return JSON {goto,query}.",
    "synthesizer_prompt":  "Answer: {query}\nContext:\n{contexts}\nIf asked for IDs, include Wikidata QIDs.",
}

ig = instrument_graph(
    graph=None,
    service_name="m1-notebook",
    trainable_keys={"planner", "executor", "synthesizer"},
    llm=stub_llm,
    initial_templates=INITIAL_TEMPLATES,
    emit_genai_child_spans=True,
    provider_name="stub",
    llm_span_name="llm.chat.completion",
    input_key="query",
    output_key="final_answer",
)

# Build and attach the graph (node funcs close over tracing_llm + templates)
ig.graph = build_graph(ig.tracing_llm, ig.templates)

print("Instrumented graph ready.")
print(f"  Templates: {sorted(ig.templates.keys())}")
print(f"  Bindings:  {sorted(ig.bindings.keys())}")
print(f"  output_key: {ig.output_key}")

In [ ]:
# --- Single invocation ---
result = ig.invoke({"query": "What is reinforcement learning?"})

print("Result keys:", sorted(result.keys()))
ans_len = len(str(result.get('final_answer', '')))
print(f"\nFinal answer ({ans_len} chars):")
print(f"  {str(result.get('final_answer', '(none)'))[:300]}")
print(f"\nPlan:")
import json as _json
try:
    print(f"  {_json.dumps(result.get('plan', {}), indent=2)[:300]}")
except Exception:
    print(f"  {str(result.get('plan', '(none)'))[:300]}")
print(f"\nContexts collected: {len(result.get('contexts', []))}")
print(f"Eval score: {result.get('eval_score', 'N/A')}")

---
## 5. Inspect OTLP Spans & `param.*` Attributes

After invocation the `TelemetrySession` holds all captured OTEL spans.
`flush_otlp()` exports them as an OTLP JSON payload.

In [ ]:
otlp = ig.session.flush_otlp(clear=True)

spans = otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]
print(f"Total spans captured: {len(spans)}\n")

# D9: Verify single trace ID per invocation
trace_ids = {s["traceId"] for s in spans}
print(f"Unique trace IDs: {len(trace_ids)} (D9: should be 1)")
assert len(trace_ids) == 1, f"Expected 1 trace ID, got {len(trace_ids)}"

# D9: Verify root invocation span exists
root_spans = [s for s in spans if s["name"].endswith(".invoke")]
assert root_spans, "Missing root invocation span (*.invoke). D9 invariant failed."
root_id = root_spans[0]["spanId"]
print(f"Root invocation span: {root_spans[0]['name']} (id={root_id[:12]}...)")
print()

for sp in spans:
    attrs = {}
    for a in sp.get("attributes", []):
        val = a.get("value", {})
        attrs[a["key"]] = val.get("stringValue", val.get("boolValue", val.get("intValue", "")))
    print(f"  Span: {sp['name']:<35} parent={sp.get('parentSpanId','(root)')[:8]}")
    for k, v in sorted(attrs.items()):
        if k.startswith("param."):
            print(f"    {k} = {str(v)[:80]}")
        elif k.startswith("gen_ai.") or k == "trace.temporal_ignore":
            print(f"    {k} = {str(v)[:80]}")
        elif k.startswith("eval.") or k.startswith("inputs.") or k.startswith("outputs."):
            print(f"    {k} = {str(v)[:80]}")
    print()

**Checkpoint:** The output above should show:
- `planner` and `synthesizer` spans with `param.<name>` and `param.<name>.trainable = True`
- Child LLM spans (configurable name, e.g. `llm.chat.completion`) with `gen_ai.*` attributes and `trace.temporal_ignore = true`

---
## 6. OTLP → TGJ → Trace Nodes

Convert the OTLP payload to **Trace-Graph JSON (TGJ)**, then ingest it
into `ParameterNode` / `MessageNode` objects — the exact format the
optimizer consumes.

In [ ]:
from opto.trace.io import otlp_traces_to_trace_json, ingest_tgj
from opto.trace.nodes import ParameterNode, MessageNode

# Re-invoke so we have fresh spans for this section
ig.invoke({"query": DEMO_QUERIES[0]})
otlp = ig.session.flush_otlp(clear=True)

# --- OTLP -> TGJ ---
docs = otlp_traces_to_trace_json(
    otlp,
    agent_id_hint="m1-notebook",
    use_temporal_hierarchy=True,
)
print(f"TGJ documents: {len(docs)}")

# --- TGJ -> Trace Nodes ---
nodes = ingest_tgj(docs[0])

# ingest_tgj stores each node under both its ID and name key,
# so deduplicate by object identity when iterating values.
param_nodes = list({id(n): n for n in nodes.values()
                    if isinstance(n, ParameterNode) and n.trainable}.values())
msg_nodes = list({id(n): n for n in nodes.values()
                  if isinstance(n, MessageNode)}.values())

print(f"\nParameterNode (trainable): {len(param_nodes)}")
for p in param_nodes:
    print(f"  {p.py_name}  trainable={p.trainable}")

# C7: Verify unique trainable param count == expected template keys
unique_param_names = set()
for p in param_nodes:
    name = p.py_name.split("/")[-1] if "/" in p.py_name else p.py_name
    unique_param_names.add(name)
print(f"\nUnique trainable params: {sorted(unique_param_names)}")

assert len(unique_param_names) == len(param_nodes), \
    f"Duplicate ParameterNodes: {len(param_nodes)} nodes but {len(unique_param_names)} unique names"
print("[OK] No duplicate ParameterNodes (C7).")

print(f"\nMessageNode: {len(msg_nodes)}")
for m in msg_nodes:
    print(f"  {m.py_name}  parents={[p.py_name.split('/')[-1] for p in m.parents]}")

# C8: Verify output node is a top-level node (not a child LLM span)
tgj_nodes = docs[0]["nodes"]
top_level_msg = []
for m in msg_nodes:
    m_name = m.py_name.split("/")[-1] if "/" in m.py_name else m.py_name
    for nid, n in tgj_nodes.items():
        if n.get("kind") == "msg" and n.get("name") == m_name:
            otel_info = (n.get("info") or {}).get("otel", {})
            is_child = str(otel_info.get("temporal_ignore", "false")).lower() in ("true", "1", "yes")
            if not is_child:
                top_level_msg.append((m, n))
            break

if top_level_msg:
    output_node, output_tgj = top_level_msg[-1]
    print(f"\nOutput node (sink): {output_node.py_name}")
    print(f"  temporal_ignore=false -> OK (not a child span)")
    print("[OK] Output node is a top-level node (C8).")
else:
    print("[WARN] No top-level message nodes found.")

In [ ]:
# --- Verify temporal chain: child spans did NOT break chaining ---
tgj_nodes = docs[0]["nodes"]

# Collect child LLM span IDs using trace.temporal_ignore marker (D10)
llm_span_ids = set()
for nid, n in tgj_nodes.items():
    otel_info = (n.get("info") or {}).get("otel", {})
    if str(otel_info.get("temporal_ignore", "false")).lower() in ("true", "1", "yes"):
        llm_span_ids.add(otel_info.get("span_id"))

print(f"Child LLM spans detected (via temporal_ignore): {len(llm_span_ids)}")
assert len(llm_span_ids) > 0, "No child LLM spans found — temporal_ignore detection failed."

# Check that no top-level node has a temporal parent pointing to a child LLM span
top_level_nodes = [
    (nid, n) for nid, n in tgj_nodes.items()
    if n.get("kind") == "msg"
    and str((n.get("info") or {}).get("otel", {}).get("temporal_ignore", "false")).lower() not in ("true", "1", "yes")
]

print(f"Top-level message nodes: {len(top_level_nodes)}")
clean = True
for nid, n in top_level_nodes:
    parent_ref = n.get("inputs", {}).get("parent", "")
    if parent_ref and ":" in parent_ref:
        _, ref_id = parent_ref.rsplit(":", 1)
        if ref_id in llm_span_ids:
            print(f"  [BUG] Node {n.get('name')} temporal parent points to child LLM span {ref_id[:12]}...")
            clean = False
        else:
            print(f"  [OK]  Node {n.get('name')} temporal parent → {ref_id[:12]}... (not a child span)")

assert clean, "Temporal parent incorrectly points to a child LLM span!"
print("\n[OK] Temporal chaining verified — no top-level node points to child spans.")

---
## 7. Bindings & `apply_updates()`

Bindings map optimizer output keys to live template values.
`apply_updates()` pushes new values through the bindings so the
**next** `invoke()` automatically uses the updated prompt.

In [ ]:
print("=" * 60)
print("BEFORE apply_updates")
print("=" * 60)
for k, b in ig.bindings.items():
    print(f"  {k}: {b.get()!r}")

# Simulate an optimizer suggesting a new planner prompt
apply_updates(
    {"planner_prompt": "Create a detailed, step-by-step plan for: {query}. Use web_researcher, wikidata_researcher, synthesizer."},
    ig.bindings,
)

print("\n" + "=" * 60)
print("AFTER apply_updates")
print("=" * 60)
for k, b in ig.bindings.items():
    print(f"  {k}: {b.get()!r}")

# Verify the change is visible in ig.templates too
assert "detailed" in ig.templates["planner_prompt"]
print("\n[OK] Binding → templates propagation verified.")

In [12]:
# Invoke again and confirm the updated template appears in the OTLP span
ig.invoke({"query": "test update"})
otlp_after = ig.session.flush_otlp(clear=True)

spans_after = otlp_after["resourceSpans"][0]["scopeSpans"][0]["spans"]
planner_sp = next(s for s in spans_after if s["name"] == "planner")
planner_attrs = {
    a["key"]: a["value"]["stringValue"] for a in planner_sp["attributes"]
}

print(f"param.planner_prompt in span:")
print(f"  {planner_attrs['param.planner_prompt']}")

assert "detailed" in planner_attrs["param.planner_prompt"]
print("\n[OK] Updated template appears in OTLP span after re-invoke.")

param.planner_prompt in span:
  Create a detailed, step-by-step plan for: {query}

[OK] Updated template appears in OTLP span after re-invoke.


In [ ]:
# Reset templates back to original for the optimization demo
apply_updates(INITIAL_TEMPLATES, ig.bindings)
print("Templates reset to original values:")
for k in sorted(INITIAL_TEMPLATES):
    print(f"  {k}: {ig.templates[k]!r}")

---
## 8. `optimize_graph()` — StubLLM End-to-End

Run the full optimization loop with **StubLLM** (deterministic, no API
calls). This verifies the complete pipeline:

```
instrument → invoke → flush OTLP → TGJ → ingest → optimizer → apply_updates
```

We use a simple length-based `eval_fn` and a mock optimizer to
demonstrate prompt value changes across iterations.

In [ ]:
from opto.trace.io import optimize_graph, EvalResult

# ---- Mock optimizer (returns deterministic updates) ----
class MockOptimizer:
    def __init__(self, param_nodes=None, **kw):
        self.calls = []
    def zero_feedback(self):
        self.calls.append("zero_feedback")
    def backward(self, output_node, feedback_text):
        self.calls.append("backward")
    def step(self):
        self.calls.append("step")
        return {
            "planner_prompt": "Create a thorough, step-by-step JSON plan for: {query}. Use web_researcher, wikidata_researcher, synthesizer.",
        }

# ---- Eval_fn: prefer evaluator score produced by the graph; fallback to structure scoring ----
def stub_eval_fn(payload):
    result = payload.get("result") or {}
    ans = str(payload.get("answer", "") or "")
    if ans.strip().startswith("[ERROR]") or not ans.strip():
        return EvalResult(score=0.0, feedback="LLM failure/empty answer")

    plan = {}
    if isinstance(result, dict):
        plan = result.get("plan", {}) or {}
    plan_steps = len(list(plan.keys())) if isinstance(plan, dict) else 0

    # Score: base + reward plan richness (up to 3 steps) + small reward for length
    score = 0.2 + 0.2 * min(plan_steps, 3) + min(len(ans) / 1200.0, 0.15)
    score = min(score, 0.95)
    return EvalResult(score=score, feedback=f"plan_steps={plan_steps}, score={score:.2f}")

print("Mock optimizer and eval_fn ready.")

In [ ]:
# -- Use the same 3 queries as the reference demo --
QUERIES = DEMO_QUERIES

mock_opt = MockOptimizer()

print("=" * 60)
print("TEMPLATE BEFORE OPTIMIZATION")
print("=" * 60)
print(f"  planner_prompt: {ig.templates['planner_prompt']!r}")
print()

opt_result = optimize_graph(
    ig,
    queries=QUERIES,
    iterations=2,
    optimizer=mock_opt,
    eval_fn=stub_eval_fn,
    apply_updates_flag=True,
)

print("\n" + "=" * 60)
print("TEMPLATE AFTER OPTIMIZATION")
print("=" * 60)
print(f"  planner_prompt: {ig.templates['planner_prompt']!r}")

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"  Baseline score:  {opt_result.baseline_score:.4f}")
print(f"  Best score:      {opt_result.best_score:.4f}")
print(f"  Best iteration:  {opt_result.best_iteration}")
print(f"  Score history:   {[round(s, 4) for s in opt_result.score_history]}")
print(f"  Optimizer calls: {mock_opt.calls}")
print(f"  Final params:    {list(opt_result.final_parameters.keys())}")
print(f"  Best params:     {list(opt_result.best_parameters.keys())}")
print(f"  Best updates:    {list(opt_result.best_updates.keys())}")

In [ ]:
# ---- Verify M1 acceptance: template changed between iter 0 and final ----
assert ig.templates["planner_prompt"] != INITIAL_TEMPLATES["planner_prompt"], \
    "Prompt should have changed after optimization!"
assert "step-by-step" in ig.templates["planner_prompt"].lower(), \
    f"Expected 'step-by-step' in optimized planner_prompt, got: {ig.templates['planner_prompt']!r}"

# Verify OTLP data present in all runs
for i, runs in enumerate(opt_result.all_runs):
    for r in runs:
        assert "resourceSpans" in r.otlp, f"Run in iter {i} missing OTLP data"

# Verify non-saturating scoring
assert opt_result.best_score < 1.0, \
    f"Score should not saturate at 1.0: {opt_result.best_score:.4f}"
assert opt_result.best_score >= opt_result.baseline_score, \
    f"Optimization should not degrade: best={opt_result.best_score:.4f} baseline={opt_result.baseline_score:.4f}"

improvement = opt_result.best_score - opt_result.baseline_score
if improvement > 0:
    print(f"[OK] Score improved by {improvement:.4f}")
else:
    print(f"[INFO] Scores equal (baseline already near cap): best={opt_result.best_score:.4f}")

print("[OK] StubLLM end-to-end optimization verified!")
print("  - Template changed across iterations")
print("  - All runs contain OTLP data")
print(f"  - Score: baseline={opt_result.baseline_score:.4f}, best={opt_result.best_score:.4f} (non-saturating)")
print("  - Optimizer was called (zero_feedback -> backward -> step)")
print("  - apply_updates propagated to bindings")

# Print optimization table
print("\n" + "=" * 60)
print("OPTIMIZATION TABLE")
print("=" * 60)
print(f"{'Iter':<6} {'Avg Score':<12} {'Best Score':<12} {'Best Iter':<12} {'Best Params'}")
print("-" * 60)
best_so_far = float("-inf")
best_iter_so_far = 0
for i, sc in enumerate(opt_result.score_history):
    if sc > best_so_far:
        best_so_far = sc
        best_iter_so_far = i
    bp = list(opt_result.best_parameters.keys()) if i == opt_result.best_iteration else []
    print(f"{i:<6} {sc:<12.4f} {best_so_far:<12.4f} {best_iter_so_far:<12} {bp}")

---
## 9. Live LLM Mode (OpenRouter)

This section runs the same pipeline against a **real LLM provider**
(OpenRouter). It is **automatically skipped** if no API key is available.

Constraints per M1 acceptance:
- Tiny dataset (≤3 items)
- Deterministic settings (`temperature=0`)
- Budget guard (`max_tokens=256` per call)

In [17]:
import requests

class OpenRouterLLM:
    """Minimal OpenRouter client (OpenAI-compatible interface).

    A1: On HTTP errors, this class now **raises** instead of converting
    the error to assistant content.  TracingLLM will catch and re-raise
    as LLMCallError so the caller can score the run as 0.
    """

    def __init__(self, api_key, model, base_url, *, max_tokens=256, temperature=0):
        self.api_key = api_key
        self.model = model
        self.base_url = base_url
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.call_count = 0

    def __call__(self, messages=None, **kwargs):
        self.call_count += 1
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        # A1: Let HTTP errors propagate — do NOT wrap them as content
        resp = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers, json=payload, timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()

        return self._wrap(data)

    @staticmethod
    def _wrap(data):
        class _M:
            pass
        class _C:
            pass
        class _R:
            pass
        r = _R()
        r.choices = []
        for c in data.get("choices", [{"message": {"content": ""}}]):
            ch = _C()
            m = _M()
            m.content = c.get("message", {}).get("content", "")
            ch.message = m
            r.choices.append(ch)
        return r

print("OpenRouterLLM class ready.")

OpenRouterLLM class ready.


In [ ]:
from opto.trace.io import LLMCallError

if not HAS_API_KEY:
    print("[SKIP] No OPENROUTER_API_KEY — live mode skipped.")
    print("       To enable: add the key in Colab Secrets or a .env file.")
    live_ok = False
else:
    print("=" * 60)
    print("LIVE LLM MODE (OpenRouter)")
    print("=" * 60)

    live_llm = OpenRouterLLM(
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,
        base_url=OPENROUTER_BASE_URL,
        max_tokens=MAX_TOKENS_PER_CALL,
        temperature=LIVE_TEMPERATURE,
    )

    live_templates = dict(INITIAL_TEMPLATES)

    live_ig = instrument_graph(
        graph=None,
        service_name="m1-live",
        trainable_keys={"planner", "executor", "synthesizer"},
        llm=live_llm,
        initial_templates=live_templates,
        emit_genai_child_spans=True,
        provider_name="openrouter",
        llm_span_name="openrouter.chat.completion",
        input_key="query",
        output_key="final_answer",
    )
    live_graph = build_graph(live_ig.tracing_llm, live_ig.templates)
    live_ig.graph = live_graph

    live_ok = False
    try:
        live_result = live_ig.invoke({"query": "What is gradient descent?"})
        ans = str(live_result.get("final_answer", "") or "")
        if ans.startswith("[ERROR]") or not ans.strip():
            print(f"[FAIL] Live LLM returned error or empty: {ans[:200]}")
        else:
            print(f"\nLive answer ({len(ans)} chars):")
            print(f"  {ans[:300]}")

            live_otlp = live_ig.session.flush_otlp(clear=False)
            live_spans = live_otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]
            trace_ids = {s["traceId"] for s in live_spans}
            has_root = any(str(sp.get("name","")).endswith(".invoke") for sp in live_spans)

            print(f"\nSpans captured: {len(live_spans)}  unique_trace_ids={len(trace_ids)}  has_root_invoke={has_root}")

            # Verify trace invariants
            if len(trace_ids) != 1:
                print(f"[WARN] Expected single trace ID, got {len(trace_ids)}")
            if not has_root:
                print("[WARN] No root *.invoke span found")

            # Check provider metadata
            for sp in live_spans:
                for a in sp.get("attributes", []):
                    if a["key"] == "gen_ai.provider.name":
                        prov = a["value"].get("stringValue", "")
                        print(f"  gen_ai.provider.name = {prov}")
                        if prov != "openrouter":
                            print(f"  [WARN] Expected 'openrouter', got '{prov}'")

            live_ok = True
            print("\n[OK] Live LLM trace validated!")

    except LLMCallError as e:
        print(f"\n[FAIL] LLMCallError during live invocation: {e}")
        print("  Skipping live optimization. Score = 0.")
    except Exception as e:
        print(f"\n[FAIL] Unexpected error during live invocation: {e}")
        print("  Skipping live optimization.")

In [ ]:
if HAS_API_KEY and live_ok:
    LIVE_QUERIES = DEMO_QUERIES[:2]

    print("=" * 60)
    print(f"LIVE OPTIMIZATION (1 iteration, {len(LIVE_QUERIES)} queries)")
    print("=" * 60)

    # Reset templates for a fresh optimization
    apply_updates(INITIAL_TEMPLATES, live_ig.bindings)
    print(f"  planner_prompt BEFORE: {live_ig.templates['planner_prompt']!r}")

    live_mock_opt = MockOptimizer()

    live_opt_result = optimize_graph(
        live_ig,
        queries=LIVE_QUERIES,
        iterations=1,
        optimizer=live_mock_opt,
        eval_fn=stub_eval_fn,
        apply_updates_flag=True,
    )

    print(f"\n  planner_prompt AFTER:  {live_ig.templates['planner_prompt']!r}")
    print(f"  Baseline score: {live_opt_result.baseline_score:.4f}")
    print(f"  Best score:     {live_opt_result.best_score:.4f}")
    print(f"  Score history:  {[round(s, 4) for s in live_opt_result.score_history]}")
    print(f"  Total LLM calls: {live_llm.call_count}")

    # --- Live OTLP inspection ---
    live_otlp_final = live_ig.session.flush_otlp(clear=True)
    try:
        live_spans = live_otlp_final["resourceSpans"][0]["scopeSpans"][0]["spans"]
        trace_ids = {s["traceId"] for s in live_spans}
        has_root = any(str(sp.get("name","")).endswith(".invoke") for sp in live_spans)
        print(f"\n  Live OTLP: {len(live_spans)} spans, {len(trace_ids)} trace IDs, root_invoke={has_root}")
    except (KeyError, IndexError) as e:
        print(f"\n  [WARN] Could not inspect live OTLP: {e}")
else:
    if not HAS_API_KEY:
        print("[SKIP] No API key — live optimization skipped.")
    else:
        print("[SKIP] Live invocation failed — live optimization skipped.")

---
## 10. Save Artifacts

Save OTLP traces, TGJ documents, and optimization summary to the run
folder (Google Drive on Colab, local fallback).

In [20]:
print("=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

# --- Save StubLLM optimization traces ---
if opt_result.all_runs and opt_result.all_runs[0]:
    # Sample trace
    sample_otlp = opt_result.all_runs[0][0].otlp
    p = os.path.join(RUN_FOLDER, "stub_sample_otlp.json")
    with open(p, "w") as f:
        json.dump(sample_otlp, f, indent=2)
    print(f"  {p}")

    # All optimization traces
    all_traces = []
    for iter_idx, runs in enumerate(opt_result.all_runs):
        label = "baseline" if iter_idx == 0 else f"iteration_{iter_idx}"
        for ri, run in enumerate(runs):
            all_traces.append({
                "iteration": label,
                "query_index": ri,
                "score": run.score,
                "otlp": run.otlp,
            })
    p = os.path.join(RUN_FOLDER, "stub_all_traces.json")
    with open(p, "w") as f:
        json.dump(all_traces, f, indent=2)
    print(f"  {p}  ({len(all_traces)} traces)")

    # TGJ from first run
    tgj_docs = otlp_traces_to_trace_json(
        sample_otlp, agent_id_hint="m1-notebook", use_temporal_hierarchy=True,
    )
    p = os.path.join(RUN_FOLDER, "stub_sample_tgj.json")
    with open(p, "w") as f:
        json.dump(tgj_docs, f, indent=2)
    print(f"  {p}")

# --- Summary ---
summary = {
    "mode": "stub",
    "baseline_score": opt_result.baseline_score,
    "best_score": opt_result.best_score,
    "best_iteration": opt_result.best_iteration,
    "score_history": opt_result.score_history,
    "final_parameters": opt_result.final_parameters,
}
p = os.path.join(RUN_FOLDER, "stub_summary.json")
with open(p, "w") as f:
    json.dump(summary, f, indent=2)
print(f"  {p}")

# --- Save live traces if available ---
if HAS_API_KEY and 'live_opt_result' in dir():
    live_traces = []
    for iter_idx, runs in enumerate(live_opt_result.all_runs):
        label = "baseline" if iter_idx == 0 else f"iteration_{iter_idx}"
        for ri, run in enumerate(runs):
            live_traces.append({
                "iteration": label,
                "query_index": ri,
                "score": run.score,
                "otlp": run.otlp,
            })
    p = os.path.join(RUN_FOLDER, "live_all_traces.json")
    with open(p, "w") as f:
        json.dump(live_traces, f, indent=2)
    print(f"  {p}  ({len(live_traces)} traces)")

    live_summary = {
        "mode": "live",
        "model": OPENROUTER_MODEL,
        "baseline_score": live_opt_result.baseline_score,
        "best_score": live_opt_result.best_score,
        "score_history": live_opt_result.score_history,
        "final_parameters": live_opt_result.final_parameters,
        "total_llm_calls": live_llm.call_count,
    }
    p = os.path.join(RUN_FOLDER, "live_summary.json")
    with open(p, "w") as f:
        json.dump(live_summary, f, indent=2)
    print(f"  {p}")

print(f"\nAll artifacts saved to: {RUN_FOLDER}")

SAVING ARTIFACTS
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_sample_otlp.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_all_traces.json  (9 traces)
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_sample_tgj.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_summary.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\live_all_traces.json  (4 traces)
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\live_summary.json

All artifacts saved to: H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1


---
## Summary

This notebook demonstrated the full M1 pipeline:

1. **`instrument_graph()`** — one-liner to add OTEL tracing to a LangGraph
2. **`param.*` attributes** — spans carry trainable prompt values
3. **OTLP → TGJ → `ParameterNode` + `MessageNode`** — optimizer-compatible trace graph
4. **Temporal integrity** — child `gen_ai.*` spans don't break chaining
5. **`apply_updates()`** — bindings propagate optimizer output to live templates
6. **`optimize_graph()`** — end-to-end loop (StubLLM deterministic + live provider)
7. **Artifacts persisted** — OTLP JSON, TGJ JSON, and summaries saved to disk

All verifications passed with StubLLM (CI-safe, deterministic). When
`OPENROUTER_API_KEY` is set, the live section additionally proves
real-provider tracing with `param.*` and `gen_ai.*` attributes.